<a href="https://colab.research.google.com/github/tamaracariappa/RAG-Pipeline-Implementing-MICE/blob/main/Major_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
BASE_PATH = "/content/drive/MyDrive/RAG_Project"

DATA_PATH = f"{BASE_PATH}/Facility Management Unified Classification Database (FMUCD).csv"

PREPROCESSED_PATH = f"{BASE_PATH}/preprocessed.csv"

CLEANED_PATH = f"{BASE_PATH}/preprocessed_clean.csv"

In [ ]:
import os
import re
import csv
import numpy as np
import pandas as pd

from tqdm import tqdm
tqdm.pandas()

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path, low_memory=False)
    return df


def normalize_text(text):

    if text is None or pd.isna(text):
        return ""

    text = str(text)

    text = text.replace("\n", " ").replace("\r", " ")

    text = text.replace('"', '')
    text = text.replace("'", "")

    text = text.replace(",", " ")
    text = text.replace(";", " ")
    text = text.replace("|", " ")
    text = text.replace("/", " ")

    text = text.lower().strip()

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'[^\w\s\-.]', '', text)

    return text


def preprocess_row(row):

    preprocessed = {

        "BuildingID": str(row.get("BuildingID", "")),

        "BuildingName": normalize_text(row.get("BuildingName", "")),

        "Type": normalize_text(row.get("Type", "")),

        "WOID": str(row.get("WOID", "")),

        "WODescription": normalize_text(row.get("WODescription", "")),

        "WOStartDate": str(row.get("WOStartDate", "")),

        "WOEndDate": str(row.get("WOEndDate", "")),

        "equipment": normalize_text(row.get("SystemDescription", ""))
    }

    for key in preprocessed:
        if preprocessed[key] is None or pd.isna(preprocessed[key]):
            preprocessed[key] = ""

    return preprocessed

In [ ]:
EXPECTED_COLUMNS = [
    "BuildingID",
    "BuildingName",
    "Type",
    "WOID",
    "WODescription",
    "WOStartDate",
    "WOEndDate",
    "equipment"
]


def load_preprocessed_csv(path):

    df = pd.read_csv(
        path,
        dtype=str,
        engine="python",
        on_bad_lines="skip"
    )

    return df.fillna("")


def build_building_reference(df):

    # 1. Strip hidden spaces before applying the regex
    clean_ids = df["BuildingID"].astype(str).str.strip()
    valid_format_mask = clean_ids.str.match(r'^[A-Z]\d{3}$', na=False)

    # 2. Use regex to catch "" AND "   " spaces to turn them into strict NaNs
    building_reference = (
        df.loc[valid_format_mask, ["BuildingID", "BuildingName", "Type"]]
        .replace(r'^\s*$', pd.NA, regex=True)
        .dropna()
        .drop_duplicates()
    )

    return building_reference


def fill_missing_building_values(df, valid_rows):

    # 1. Identify invalid IDs using the stripped column
    clean_ids = df["BuildingID"].astype(str).str.strip()
    invalid_mask = ~clean_ids.str.match(r'^[A-Z]\d{3}$', na=False)

    missing_indices = df.index[invalid_mask]
    total_missing = len(missing_indices)

    print(f"Filling {total_missing:,} missing or invalid BuildingID values...")

    if total_missing == 0:
        return df

    ref_df = pd.DataFrame(valid_rows)

    if ref_df.empty:
        raise ValueError("No valid BuildingIDs found in the dataset to sample from.")

    # 2. Sample from our 138 clean rows to fill the gaps
    sampled = ref_df.sample(
        n=total_missing,
        replace=True
    ).reset_index(drop=True)

    # 3. Overwrite the bad data in the main dataframe
    for i, idx in enumerate(missing_indices):
        df.at[idx, "BuildingID"] = sampled.at[i, "BuildingID"]
        df.at[idx, "BuildingName"] = sampled.at[i, "BuildingName"]
        df.at[idx, "Type"] = sampled.at[i, "Type"]

    return df

def drop_unfillable_rows(df):
    """
    Converts empty strings to NaNs and drops incomplete rows
    to ensure absolute data completeness for embeddings.
    """
    initial_count = len(df)

    # Convert empty strings back to True NaNs and drop
    df = df.replace("", pd.NA).dropna().reset_index(drop=True)

    final_count = len(df)
    dropped = initial_count - final_count

    print(f"Dropped {dropped:,} incomplete rows.")
    print(f"Final clean row count: {final_count:,}")

    return df

def clean_preprocessed_dataset(input_path, output_path):

    print("\nCleaning dataset...")
    df = load_preprocessed_csv(input_path)

    # FORCE strip all IDs immediately upon loading to prevent downstream mismatches
    df["BuildingID"] = df["BuildingID"].astype(str).str.strip()

    valid_rows = build_building_reference(df)
    df = fill_missing_building_values(df, valid_rows)
    df = handle_remaining_missing_values(df)

    # --- NEW STEP INSERTED HERE ---
    df = drop_unfillable_rows(df)

    df = enforce_schema(df)

    save_clean_csv(df, output_path)

    print("Cleaning complete.")
    return df

def handle_remaining_missing_values(df):

    df["WOEndDate"] = df["WOEndDate"].replace("", "ONGOING")

    df = df.fillna("")

    return df


def enforce_schema(df):

    for col in EXPECTED_COLUMNS:

        if col not in df.columns:
            df[col] = ""

    return df[EXPECTED_COLUMNS]


def save_clean_csv(df, output_path):

    df.to_csv(
        output_path,
        index=False,
        quoting=csv.QUOTE_ALL
    )

    print(f"Saved → {output_path}")



In [ ]:
if os.path.exists(PREPROCESSED_PATH):

    print("preprocessed.csv already exists")

else:

    df = load_dataset(DATA_PATH)

    print(f"Loaded {len(df)} rows")

    records = []

    for _, row in tqdm(df.iterrows(), total=len(df)):

        records.append(preprocess_row(row))

    processed_df = pd.DataFrame(records)

    processed_df.to_csv(PREPROCESSED_PATH, index=False)

    print("Saved preprocessed.csv")

preprocessed.csv already exists


In [ ]:
"""if os.path.exists(CLEANED_PATH):
    clean_df = load_dataset(CLEANED_PATH)
    print("preprocessed_clean.csv already exists")

else:
"""
clean_df = clean_preprocessed_dataset(
        PREPROCESSED_PATH,
        CLEANED_PATH
    )


Cleaning dataset...
Filling 3,594,594 missing or invalid BuildingID values...
Dropped 12,576 incomplete rows.
Final clean row count: 3,718,866
Saved → /content/drive/MyDrive/RAG_Project/preprocessed_clean.csv
Cleaning complete.


In [ ]:
pd.set_option('display.max_rows', None)
clean_df.head(n=330)

,BuildingID,BuildingName,Type,WOID,WODescription,WOStartDate,WOEndDate,equipment
0,A050,coburg road 6414,research,WO244376,yearly inspection of safety relief valves,2012-12-17 07:49:05,2015-12-07 12:41:28,plumbing
1,E800,glengary,student experience,WO245713,review steam system throughout campus for emer...,2013-01-02 11:31:55,2017-03-22 10:51:59,hvac
2,C201,lsc-biolearth,mixed teaching research,WO246478,yearly refrigeration inspection - zone 3,2013-01-07 09:22:45,2015-05-05 10:30:36,hvac
3,J351,sexton memorial gym,student experience,WO247428,cppainter - j building - gym - repair damaged ...,2013-01-15 09:05:33,2015-05-11 11:03:50,interior finishes
4,C201,lsc-biolearth,mixed teaching research,WO248110,air flow too low or too high,2013-01-21 09:42:39,2016-01-18 13:46:15,hvac
5,N160,jenkins hall,student experience,WO248898,steam chambers - monthly inspection,2013-01-25 13:59:50,2015-03-09 14:00:38,hvac
6,J920,morris 5247,research,WO249443,replace fan 5247 morris st,2013-01-29 09:27:12,2016-02-04 14:14:40,hvac
7,C202,lsc-oceanograph,mixed teaching research,WO250777,check heating system,2013-02-07 15:02:04,2016-03-10 09:52:34,hvac
8,M440,hancock veterinary building,other,WO254349,assist state chemicals,2013-03-13 12:03:07,2016-03-16 09:42:13,hvac
9,E100,stud. union bldg,student experience,WO256011,tile missing by door in front of entry,2013-03-25 13:22:17,2015-07-16 07:22:41,interior finishes


In [ ]:
pattern = r"^[A-Z]\d{3}$"

invalid_mask = ~clean_df["BuildingID"].astype(str).str.match(pattern, na=False)

num_invalid = invalid_mask.sum()
print("Number of invalid BuildingIDs:", num_invalid)

# Select only the first two columns of the invalid rows
invalid_rows = clean_df.loc[invalid_mask, clean_df.columns[:2]]

print(invalid_rows.head(n = 300))

Number of invalid BuildingIDs: 0
Empty DataFrame
Columns: [BuildingID, BuildingName]
Index: []


In [ ]:
clean_df.isnull().sum()

,0
BuildingID,0
BuildingName,0
Type,0
WOID,0
WODescription,0
WOStartDate,0
WOEndDate,0
equipment,0


In [ ]:
(clean_df == "").sum()

,0
BuildingID,0
BuildingName,0
Type,0
WOID,0
WODescription,0
WOStartDate,0
WOEndDate,0
equipment,0


In [ ]:
!pip install -q sentence-transformers